# Start here: final BBO capstone report

## tl;dr

- The canonical ledger contains **104 verified query/return pairs**: 13 rounds for each of eight functions.
- **12 returns changed an incumbent**. Week 13 improved F5 and F6 after its eight proposals had been frozen from the 96-pair Week 12 boundary.
- Rolling chronological GP validation now contains **104 held-out predictions** (13 per function). All eight functions beat the historical-mean RMSE baseline, but calibration remains uneven.
- The function-specific UCB/EI/PI policy was adaptive and heuristic, not a statistically controlled acquisition comparison.

This is the single reader-first synthesis. The weekly folders remain historical records; the canonical source map is in `Documentation/ARTEFACT_GUIDE.md`.

## Context & Methods

Eight unknown functions were maximised sequentially. Each round added one portal-returned observation per function. Gaussian Process surrogates and acquisition functions increasingly replaced manual exploration, while validation, provenance, and submission-format checks became stricter.

### Key assumptions and limitations

- Objective scales differ, so performance is compared within functions rather than pooled.
- The chronological folds respect time order but arise from an adaptive campaign, not an independent test set.
- Incumbent change means a strictly larger verified output; it is not proof of a global optimum.
- Week 13 acquisition choices were recorded before Week 13 outcomes. Their realised results do not make the policy a controlled experiment.
- High-dimensional functions remain sparsely covered, and several GP hyperparameters reach configured bounds.
- Predictive calibration is imperfect, especially for F5–F7, so uncertainty estimates should not be treated as exact probabilities.
- Source-file modification dates are provenance metadata, not authoritative portal submission times.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'Results/query_output_ledger.csv').is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Repository root not found')

ledger = pd.read_csv(ROOT / 'Results/query_output_ledger.csv')
scoreboard = pd.read_csv(ROOT / 'Final_Report/final_scoreboard.csv')
timeline = pd.read_csv(ROOT / 'Final_Report/incumbent_timeline.csv')
strategy = pd.read_csv(ROOT / 'Final_Report/function_strategy_table.csv')
metrics = pd.read_csv(ROOT / 'Results/gp_validation_metrics.csv')

assert len(ledger) == 104 and ledger.groupby('function').size().eq(13).all()
assert len(scoreboard) == len(strategy) == len(metrics) == 8
assert scoreboard['incumbent_changes'].sum() == 12
assert scoreboard.loc[scoreboard.week_13_improved, 'function'].tolist() == ['F5', 'F6']
print('Integrity checks passed: 104 returns, 8 functions, 12 incumbent changes.')

Integrity checks passed: 104 returns, 8 functions, 12 incumbent changes.


## Data

The scoreboard and timeline below are derived from `Results/performance_summary_weeks_01_to_13.csv`, itself generated from the checksum-backed canonical ledger. The strategy table is derived from the frozen Week 13 strategy summary.

## Results

### Final scoreboard

In [2]:
view = scoreboard[['function', 'dimensions', 'initial_incumbent', 'final_incumbent', 'incumbent_changes', 'change_weeks', 'week_13_improved', 'week_13_acquisition', 'rolling_rmse_skill', 'coverage_95']]
display(view.style.format({'initial_incumbent': '{:.6g}', 'final_incumbent': '{:.6g}', 'rolling_rmse_skill': '{:.3f}', 'coverage_95': '{:.1%}'}))

,function,dimensions,initial_incumbent,final_incumbent,incumbent_changes,change_weeks,week_13_improved,week_13_acquisition,rolling_rmse_skill,coverage_95
0,F1,2,7.71088e-16,7.71088e-16,0,nan,False,UCB,0.303,100.0%
1,F2,2,0.611205,0.611205,0,nan,False,EI,0.342,100.0%
2,F3,3,-0.0348353,-0.0226293,1,12,False,PI,0.379,92.3%
3,F4,4,-4.02554,0.369975,2,"3, 12",False,UCB,0.774,76.9%
4,F5,4,1088.86,4440.56,3,"8, 12, 13",True,EI,0.615,61.5%
5,F6,5,-0.714265,-0.405993,2,"12, 13",True,EI,0.617,76.9%
6,F7,6,1.36497,2.2668,2,"3, 12",False,UCB,0.594,69.2%
7,F8,8,9.59848,9.9399,2,"1, 2",False,UCB,0.909,100.0%


### Campaign timeline: when incumbents changed

In [3]:
fig, ax = plt.subplots(figsize=(12, 5.5))
for index, function in enumerate([f'F{i}' for i in range(1, 9)]):
    points = timeline[timeline.function.eq(function)]
    ax.hlines(index, 0, 13, color='#CBD5E1', linewidth=1)
    ax.scatter(points.week, [index] * len(points), s=85, color='#1D4ED8', edgecolor='white', zorder=3)
    for row in points.itertuples(index=False):
        label = 'start' if row.week == 0 else f'W{row.week}'
        ax.annotate(label, (row.week, index), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8)
ax.set(yticks=range(8), yticklabels=[f'F{i}' for i in range(1, 9)], xticks=range(14), xlabel='Campaign week (0 = starter data)', title='Verified incumbent-change timeline')
ax.invert_yaxis(); ax.spines[['top', 'right', 'left']].set_visible(False); ax.grid(axis='x', alpha=.15)
fig.tight_layout()
display(fig)
plt.close(fig)

<Figure size 1200x550 with 1 Axes>

### Function-by-function Week 13 strategy

In [4]:
display(strategy)

,function,dimensions,acquisition,kappa,xi_output_sd_fraction,candidate_focus,pre_outcome_rationale
0,F1,2,UCB,3.00,0.010,sobol,Sparse near-zero responses still justify uncer...
1,F2,2,EI,2.00,0.020,local,The strong Week 12 return supports balanced im...
2,F3,3,PI,1.50,0.005,sobol,The new best reinforces a stable structure and...
3,F4,4,UCB,2.75,0.020,local,"The large Week 12 improvement is promising, wh..."
4,F5,4,EI,1.50,0.010,local,The much stronger Week 12 incumbent warrants f...
5,F6,5,EI,1.75,0.010,local,The Week 12 best supports exploitation with un...
6,F7,6,UCB,2.00,0.010,local,The new best supports local focus while modera...
7,F8,8,UCB,3.00,0.020,sobol,A near-best Week 12 result does not remove eig...


### How the final workflow connects

```mermaid
flowchart LR
  A[Weekly inputs and portal returns] --> B[Canonical 104-pair ledger]
  B --> C[Per-function GP models]
  B --> D[Chronological rolling evaluation]
  D --> E[Calibration and hyperparameter diagnostics]
  C --> F[Week 13 acquisition policy: UCB / EI / PI]
  E --> F
  F --> G[Candidate generator]
  G --> H[Six-decimal portal validation]
  H --> I[Frozen Week 13 proposals]
  I -. prospective returns .-> B
```

The dashed edge is important: Week 13 outcomes entered the ledger only after the proposals were frozen.

## Takeaways

1. **Optimisation:** F5 achieved the largest absolute gain on its own scale; F3, F4, F6, F7, and F8 also finished above their starter incumbents. F1 and F2 never exceeded strong starter values.
2. **Learning:** model-guided search improved the campaign, but function-specific behaviour justified different acquisition choices rather than one universal setting.
3. **Evaluation:** positive RMSE skill across all functions supports using the GPs as predictive aids, while uneven interval coverage argues against overconfident interpretation.
4. **Evidence discipline:** the proposal freeze, canonical ledger, derived evaluation, checksums, and historical archive separate decisions from later outcomes.
5. **Reproducibility:** run `Code/run_frozen_repository.py` for the complete pipeline or follow the shorter public guide in `REPRODUCIBILITY.md`.